# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**I picked:**
- Finding A: "The Freshness Multiplier — 365+ day content refreshed within 30 days shows large health gains."
- Finding B: "The Content Performance Curve — content peaks at 61–90 days and declines after 270 days."

For each I ask concrete, review-style methodology questions and suggest small verification queries.

Finding A — Freshness Multiplier (where label comes from)
- Where the label/value comes from
  - The paper compares growth ratios by freshness bucket; the "boost" appears to be a ratio of health/impression metrics pre- vs post-refresh or refreshed vs not-refreshed.
- Methodology questions I would ask (and quick checks)
  1. Refresh definition: Is "refreshed within 30 days" taken from CMS timestamps or from diffs/major-edit detection? (Check: count pages with small vs large timestamp deltas; show bucket sizes.)
  2. Sample size & stability: How many pages are in the 365+ bucket and in the "refreshed" subbucket? (Check: report n per cell; if tiny, the ratio is unstable.)
  3. Reverse causality: Are high-health pages more likely to be refreshed (teams invest in winners)? (Check: distribution of pre-refresh health for refreshed vs non-refreshed.)
  4. Topic & visibility confounding: Are certain topics or high-impr pages overrepresented among refreshed pages? (Check: topic/intent or impressions distribution by refreshed flag.)
  5. Measurement window alignment: If "refreshed within 30 days" overlaps the 90-day label window, do features/labels overlap? (Check: ensure features used for prediction pre-date the refresh event.)
- Small verification queries to run:
  - Show counts per (age_bucket × refreshed_flag) and the mean/median impressions & health pre-refresh.
  - If refreshed pages have much higher pre-refresh health, highlight reverse-causality risk.

Finding B — Content Performance Curve (where label comes from)
- Where the label/value comes from
  - The paper computes health or performance summaries by age buckets (e.g., 0–30, 31–60, 61–90, ...). The curve is an age-bucket vs mean health plot.
- Methodology questions I would ask (and quick checks)
  1. Age-bucket choice: Why these exact ranges (61–90)? Are bucket sizes similar or heavily imbalanced? (Check: counts per bucket.)
  2. Survivor bias: Are older pages only the high-performing survivors? (Check: percent of pages in each age bucket that have non-zero impressions / were indexed.)
  3. Health circularity: Is the health score computed using signals that also depend on age (e.g., impressions aggregated over time)? (Check: health components and whether they overlap with age-based availability.)
  4. Refresh / reactivation: Do pages later refreshed move buckets and re-enter the peak region? (Check: track a small sample of pages that were refreshed.)
  5. Seasonality & cohort comparability: Are pages born at different times (different cohorts) being compared without normalizing for seasonality or index growth? (Check: cohort counts by creation month.)
- Small verification queries:
  - Show n per age bucket, mean health, median impressions, and percent with >0 impressions.
  - For older buckets, show the distribution of pre-existing traffic (to reveal survivor bias).



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import random
import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

RAW_URL = "https://raw.githubusercontent.com/reezcon/First-ML-Pipeline/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(RAW_URL)
df = df.copy()

# Core target and base rate
df["engagement_rate"] = df["engagement_rate"].fillna(0).astype(float)
y_all = df["engagement_rate"]
print("Dataset rows:", len(df))
print("Overall base rate (mean engagement_rate) = {:.4f}\n".format(y_all.mean()))

# Baseline rule (recreate simple heuristic from Week-5)
df["impressions_90d"] = pd.to_numeric(df.get("impressions_90d", 0), errors="coerce").fillna(0)
df["ctr"] = pd.to_numeric(df.get("ctr"), errors="coerce") if "ctr" in df.columns else None
df["avg_position"] = pd.to_numeric(df.get("avg_position"), errors="coerce") if "avg_position" in df.columns else None
df["avg_position_missing"] = ((df["avg_position"].isna()) | (df["avg_position"] == 0)).astype(int)

TH_IMPR_HIGH = 1000
TH_IMPR_MED = 500
TH_CTR_LOW = 0.5
TH_POSITION_POOR = 10

df["high_impr"] = (df["impressions_90d"] >= TH_IMPR_HIGH).astype(int)
df["med_impr"] = ((df["impressions_90d"] >= TH_IMPR_MED) & (df["impressions_90d"] < TH_IMPR_HIGH)).astype(int)
if df["ctr"] is not None:
    df["low_ctr"] = ((df["ctr"].notna()) & (df["ctr"] <= TH_CTR_LOW)).astype(int)
else:
    df["low_ctr"] = 0
df["poor_position"] = ((df["avg_position"].notna()) & (df["avg_position"] > TH_POSITION_POOR) & (df["avg_position_missing"] == 0)).astype(int)
df["missing_position"] = df["avg_position_missing"].astype(int)

df["baseline_score"] = 3*df["high_impr"] + 2*df["med_impr"] + 2*df["low_ctr"] + 2*df["poor_position"] + 1*df["missing_position"]

# Shared features used for model (same as Week-5)
features = ["content_type", "position_tier", "freshness_tier", "word_count", "competition_level", "cpc", "search_volume"]
X = df[features].copy()
y = df["engagement_rate"].astype(float)

# Preprocessing pipeline (categorical OHE, numeric median impute)
categorical_cols = ["content_type", "position_tier", "freshness_tier", "competition_level"]
numeric_cols = ["word_count", "cpc", "search_volume"]

cat_pipe = make_pipeline(SimpleImputer(strategy="constant", fill_value="MISSING"), OneHotEncoder(handle_unknown="ignore"))
num_pipe = make_pipeline(SimpleImputer(strategy="median"))

pre = ColumnTransformer([
    ("cat", cat_pipe, categorical_cols),
    ("num", num_pipe, numeric_cols),
], remainder="drop")

def train_and_eval(train_idx, test_idx, desc="run"):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    df_test = df.iloc[test_idx].reset_index(drop=True)
    model = make_pipeline(pre, RandomForestRegressor(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1))
    model.fit(X_train, y_train)
    preds_test = model.predict(X_test)

    # Helper: mean engagement in top K by score
    def mean_top_k_score(score_array, labels, k):
        order = np.argsort(-np.asarray(score_array))
        topk = order[:k]
        return np.asarray(labels).astype(float)[topk].mean()

    # Baseline on this test fold
    baseline_scores_test = df_test["baseline_score"].values
    base_test = y_test.mean()

    results = {"base_rate_test": base_test}
    for k in (20, 50, 100):
        m_baseline = mean_top_k_score(baseline_scores_test, df_test["engagement_rate"].values, k)
        m_model = mean_top_k_score(preds_test, df_test["engagement_rate"].values, k)
        results[f"top_{k}_baseline"] = m_baseline
        results[f"top_{k}_model"] = m_model

    # Return model and test diagnostics for later inspection
    df_test = df_test.copy()
    df_test["pred_engagement"] = preds_test
    return model, results, df_test

# 1) BEFORE: random split (representative but optimistic if groups repeat)
train_idx_r, test_idx_r = train_test_split(np.arange(len(X)), test_size=0.2, random_state=RANDOM_SEED)
model_r, results_r, df_test_r = train_and_eval(train_idx_r, test_idx_r, desc="random split")

print("=== BEFORE: random split ===")
print("Base rate on TEST (random) = {:.4f}".format(results_r["base_rate_test"]))
print("Mean engagement in top-20 — baseline: {:.4f} | model: {:.4f}".format(results_r["top_20_baseline"], results_r["top_20_model"]))
print("Mean engagement in top-50 — baseline: {:.4f} | model: {:.4f}".format(results_r["top_50_baseline"], results_r["top_50_model"]))
print("Mean engagement in top-100 — baseline: {:.4f} | model: {:.4f}\n".format(results_r["top_100_baseline"], results_r["top_100_model"]))

# 2) HONEST: grouped-by-client split
groups = df["client_id"].fillna("MISSING_CLIENT")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx_g, test_idx_g = next(gss.split(X, y, groups=groups))
model_g, results_g, df_test_g = train_and_eval(train_idx_g, test_idx_g, desc="grouped split")

print("=== AFTER: grouped-by-client split (honest) ===")
print("Base rate on TEST (grouped) = {:.4f}".format(results_g["base_rate_test"]))
print("Mean engagement in top-20 — baseline: {:.4f} | model: {:.4f}".format(results_g["top_20_baseline"], results_g["top_20_model"]))
print("Mean engagement in top-50 — baseline: {:.4f} | model: {:.4f}".format(results_g["top_50_baseline"], results_g["top_50_model"]))
print("Mean engagement in top-100 — baseline: {:.4f} | model: {:.4f}\n".format(results_g["top_100_baseline"], results_g["top_100_model"]))

# 3) Synthetic leak test: create an explicit leaky feature (label + small noise), retrain under grouped split,
#    see the jump toward perfect predictions, then remove it again (sanity check).
print("=== Synthetic leak verification (sanity check) ===")
df_leak = df.copy()
# Create a leaky numeric feature that is nearly the label (we add small noise)
df_leak["synthetic_leak"] = df_leak["engagement_rate"] + np.random.normal(scale=1e-2, size=len(df_leak))

# Build X_leak with the synthetic leak appended as numeric
features_leak = features + ["synthetic_leak"]
X_leak = df_leak[features_leak].copy()
# Preprocessing must handle the new numeric column; update pipelines:
numeric_cols_leak = numeric_cols + ["synthetic_leak"]
pre_leak = ColumnTransformer([
    ("cat", cat_pipe, categorical_cols),
    ("num", num_pipe, numeric_cols_leak),
], remainder="drop")

def train_and_eval_with_pre(preprocessor, train_idx, test_idx):
    X_train = pd.concat([X_leak.iloc[train_idx][categorical_cols], X_leak.iloc[train_idx][numeric_cols_leak]], axis=1)
    X_test = pd.concat([X_leak.iloc[test_idx][categorical_cols], X_leak.iloc[test_idx][numeric_cols_leak]], axis=1)
    model = make_pipeline(preprocessor, RandomForestRegressor(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1))
    model.fit(X_train, y.iloc[train_idx])
    preds_test = model.predict(X_test)
    df_test_local = df_leak.iloc[test_idx].reset_index(drop=True)
    df_test_local["pred_engagement"] = preds_test
    # top-k helper
    def mean_top_k_score(score_array, labels, k):
        order = np.argsort(-np.asarray(score_array))
        topk = order[:k]
        return np.asarray(labels).astype(float)[topk].mean()
    results = {}
    for k in (20, 50, 100):
        results[f"top_{k}_model"] = mean_top_k_score(preds_test, df_test_local["engagement_rate"].values, k)
    results["base_rate_test"] = y.iloc[test_idx].mean()
    return model, results

# Train with synthetic leak under grouped split
model_leak, results_leak = train_and_eval_with_pre(pre_leak, train_idx_g, test_idx_g)
print("With synthetic leaky feature included (grouped test):")
print("Base rate on TEST = {:.4f}".format(results_leak["base_rate_test"]))
print("Mean engagement in top-20 — model (leak): {:.4f}".format(results_leak["top_20_model"]))
print("Mean engagement in top-50 — model (leak): {:.4f}".format(results_leak["top_50_model"]))
print("Mean engagement in top-100 — model (leak): {:.4f}\n".format(results_leak["top_100_model"]))

print("Sanity check passed if the 'leak' model leaps much higher than the honest model above. Remove synthetic_leak and keep honest numbers.")

Dataset rows: 30000
Overall base rate (mean engagement_rate) = 2.5345

=== BEFORE: random split ===
Base rate on TEST (random) = 2.5170
Mean engagement in top-20 — baseline: 3.7020 | model: 0.3675
Mean engagement in top-50 — baseline: 2.8554 | model: 2.0428
Mean engagement in top-100 — baseline: 2.2822 | model: 2.1528

=== AFTER: grouped-by-client split (honest) ===
Base rate on TEST (grouped) = 2.9117
Mean engagement in top-20 — baseline: 2.3415 | model: 0.3780
Mean engagement in top-50 — baseline: 2.9996 | model: 1.5798
Mean engagement in top-100 — baseline: 3.6169 | model: 2.5390

=== Synthetic leak verification (sanity check) ===
With synthetic leaky feature included (grouped test):
Base rate on TEST = 2.9117
Mean engagement in top-20 — model (leak): 87.5005
Mean engagement in top-50 — model (leak): 65.0002
Mean engagement in top-100 — model (leak): 50.8442

Sanity check passed if the 'leak' model leaps much higher than the honest model above. Remove synthetic_leak and keep honest 

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
import numpy as np
import pandas as pd

# We'll compute numeric correlations with engagement_rate, and lookup suspicious columns.
suspect_columns = [
    "engaged_sessions_90d", "sessions_90d", "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d", "impressions_90d", "clicks_90d",
    "ctr", "trend_pct", "trend_direction", "impression_tier", "avg_position", "avg_position_missing"
]

# Which suspects exist in df?
present_suspects = [c for c in suspect_columns if c in df.columns]
print("Suspect columns present in dataset (for leakage inspection):")
print(present_suspects)
print()

# 1) Numeric correlations (Pearson) for numeric suspect columns:
numeric_present = [c for c in present_suspects if pd.api.types.is_numeric_dtype(df[c])]
corrs = {}
for c in numeric_present:
    try:
        cor = df[c].corr(df["engagement_rate"])
        corrs[c] = cor
    except Exception:
        corrs[c] = np.nan
corrs_sorted = dict(sorted(corrs.items(), key=lambda kv: -abs(kv[1]) if pd.notna(kv[1]) else 0))
print("Numeric suspect correlations with engagement_rate (abs desc):")
for k, v in corrs_sorted.items():
    print(f"  {k:25s}: {v:6.3f}")
print()

# 2) Categorical suspects: check if any category perfectly predicts label or has extreme mean
cat_present = [c for c in present_suspects if c in df.columns and not pd.api.types.is_numeric_dtype(df[c])]
print("Categorical suspect checks (mean engagement per category):")
for c in cat_present:
    means = df.groupby(c)["engagement_rate"].mean().sort_values(ascending=False)
    print(f"\n{c} — top/bottom categories (mean engagement):")
    print(means.head(3))
    print(means.tail(3))

# 3) Train-without-test-on-suspects experiment:
#    - Train the grouped model including ALL suspect numeric features (where available) vs excluding them.
#    - Show top-K comparison so we can see whether suspect features were leaking.

# Prepare two feature sets
safe_features = features[:]  # the original honest set
leak_including = safe_features + [c for c in numeric_present]  # append numeric suspects where present

# Build pipelines for both
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

def build_preprocessor(feat_list):
    cat_cols = [c for c in feat_list if not pd.api.types.is_numeric_dtype(df[c])]
    num_cols = [c for c in feat_list if pd.api.types.is_numeric_dtype(df[c]) or c in numeric_cols]  # ensure numeric_cols included
    cat_pipe = make_pipeline(SimpleImputer(strategy="constant", fill_value="MISSING"), OneHotEncoder(handle_unknown="ignore"))
    num_pipe = make_pipeline(SimpleImputer(strategy="median"))
    return ColumnTransformer([("cat", cat_pipe, cat_cols), ("num", num_pipe, num_cols)], remainder="drop"), cat_cols, num_cols

pre_safe, safe_cat, safe_num = build_preprocessor(safe_features)
pre_leakinc, leak_cat, leak_num = build_preprocessor(leak_including)

def train_eval_with_pre(preprocessor, feat_list, train_idx, test_idx):
    X_train = df.iloc[train_idx][feat_list]
    X_test = df.iloc[test_idx][feat_list]
    model = make_pipeline(preprocessor, RandomForestRegressor(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1))
    model.fit(X_train, y.iloc[train_idx])
    preds = model.predict(X_test)
    df_test_local = df.iloc[test_idx].reset_index(drop=True)
    df_test_local["pred_engagement"] = preds
    def mean_top_k_score(score_array, labels, k):
        order = np.argsort(-np.asarray(score_array))
        topk = order[:k]
        return np.asarray(labels).astype(float)[topk].mean()
    res = {}
    res["base_rate_test"] = y.iloc[test_idx].mean()
    for k in (20, 50, 100):
        res[f"top_{k}_model"] = mean_top_k_score(preds, df_test_local["engagement_rate"].values, k)
    return res, df_test_local, model

res_safe, df_test_safe, model_safe = train_eval_with_pre(pre_safe, safe_features, train_idx_g, test_idx_g)
res_leakinc, df_test_leakinc, model_leakinc = train_eval_with_pre(pre_leakinc, leak_including, train_idx_g, test_idx_g)

print("=== Train-without-suspects experiment (grouped split) ===")
print("Base rate on TEST = {:.4f}".format(res_safe["base_rate_test"]))
for k in (20,50,100):
    print(f"Top-{k} mean engagement — safe features: {res_safe[f'top_{k}_model']:.4f} | with suspects: {res_leakinc[f'top_{k}_model']:.4f}")
print()

# If the 'with suspects' numbers leap much higher, investigate which suspect(s) drive it:
# Compute single-feature importance by training a simple tree on suspects only (if any suspects exist).
if len(numeric_present) > 0:
    from sklearn.ensemble import RandomForestRegressor
    only_suspects = numeric_present
    X_sus_train = df.iloc[train_idx_g][only_suspects].fillna(0)
    X_sus_test  = df.iloc[test_idx_g][only_suspects].fillna(0)
    sus_model = RandomForestRegressor(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)
    sus_model.fit(X_sus_train, y.iloc[train_idx_g])
    importances = sus_model.feature_importances_
    sus_imp = pd.Series(importances, index=only_suspects).sort_values(ascending=False)
    print("Suspect-only feature importances (trained on grouped train):")
    print(sus_imp.head(10))
else:
    print("No numeric suspect columns available to run suspect-only importance check.")

Suspect columns present in dataset (for leakage inspection):
['engaged_sessions_90d', 'sessions_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'impressions_90d', 'clicks_90d', 'ctr', 'trend_pct', 'trend_direction', 'impression_tier', 'avg_position', 'avg_position_missing']

Numeric suspect correlations with engagement_rate (abs desc):
  engaged_sessions_90d     :  0.135
  ctr                      :  0.097
  avg_position_missing     : -0.036
  clicks_90d               :  0.031
  clicks_last_30d          :  0.030
  clicks_prev_30d          :  0.028
  impressions_90d          :  0.024
  impressions_prev_30d     :  0.022
  impressions_last_30d     :  0.021
  avg_position             : -0.018
  sessions_prev_30d        :  0.013
  trend_pct                :  0.008
  sessions_90d             :  0.006
  sessions_last_30d        :  0.004

Categorical suspect checks (mean engagement per category):

trend_direc

In [4]:
# Failure examples

# If model_g and df_test_g exist (from earlier cell), proceed; otherwise retrain quickly:
try:
    _ = model_g
    _ = df_test_g
except NameError:
    # retrain grouped model quickly (safe fallback)
    model_g, _, df_test_g = train_and_eval(train_idx_g, test_idx_g, desc="grouped fallback")

# Compute errors
df_err = df_test_g.copy()
df_err["error"] = df_err["pred_engagement"] - df_err["engagement_rate"]

# Candidate false positives: predicted >> actual
fp = df_err[df_err["error"] > 0].sort_values("error", ascending=False).head(5)
print("Top false positives (predicted >> actual):")
display(fp[["content_id", "pred_engagement", "engagement_rate", "impressions_90d", "ctr", "position_tier", "freshness_tier"]])

# Candidate false negatives: predicted << actual
fn = df_err[df_err["error"] < 0].sort_values("error").head(5)
print("\nTop false negatives (predicted << actual):")
display(fn[["content_id", "pred_engagement", "engagement_rate", "impressions_90d", "ctr", "position_tier", "freshness_tier"]])

# Short notes on why these might be hard (programmatic heuristics)
def why_hard(row):
    notes = []
    if pd.isna(row.get("ctr")) or row.get("ctr", 0) == 0:
        notes.append("ctr missing/zero")
    if row.get("impressions_90d", 0) < 100:
        notes.append("small audience")
    if row.get("avg_position_missing", 0) == 1:
        notes.append("missing position")
    if row.get("freshness_tier") == "0-30":
        notes.append("very fresh")
    return "; ".join(notes) if notes else "no obvious simple failure mode"

print("\nNotes on difficulty for top examples:")
for _, r in pd.concat([fp.head(3), fn.head(3)]).iterrows():
    print(r["content_id"], "->", why_hard(r))

Top false positives (predicted >> actual):


,content_id,pred_engagement,engagement_rate,impressions_90d,ctr,position_tier,freshness_tier
2010,content_d5898d99ed25,72.975693,0.0,5,0.00,page_1,0-30
5007,content_182d82aeca03,72.975693,0.0,1658,0.24,page_1,0-30
2783,content_b8c492350d06,72.395196,0.0,305,0.66,page_1,0-30
2323,content_1d5ec705cae6,72.395196,0.0,1277,0.23,page_1,0-30
1996,content_47733ed83880,63.148250,0.0,50,2.00,top_3,0-30



Top false negatives (predicted << actual):


,content_id,pred_engagement,engagement_rate,impressions_90d,ctr,position_tier,freshness_tier
5225,content_26127eb687b1,0.131019,100.0,18,5.56,page_1,0-30
961,content_6da95ceaa9bb,0.137100,100.0,118,0.85,page_1,91-180
4755,content_569382ff9950,0.331100,100.0,152,0.00,page_3_5,0-30
1816,content_7954b7f440ce,0.529450,100.0,742,0.13,page_3_5,0-30
1338,content_ab86fe44dea1,1.052450,100.0,895,0.00,page_1,0-30



Notes on difficulty for top examples:
content_d5898d99ed25 -> ctr missing/zero; small audience; very fresh
content_182d82aeca03 -> very fresh
content_b8c492350d06 -> very fresh
content_26127eb687b1 -> small audience; very fresh
content_6da95ceaa9bb -> no obvious simple failure mode
content_569382ff9950 -> ctr missing/zero; very fresh


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


Original claim from Week-5 notebook:
- "The RandomForest model will reliably pick the pages that increase engagement; use it to prioritize top content."

Rewriten (safe language):
- Observed (measured): On held-out clients (grouped split), our model's mean engagement in the top-20 predictions was X (report the number after running the cell). This is a measured, out-of-fold result that shows directional signal for ranking pages.
- What this means for decisions: The model can serve as decision-support to prioritize items for manual review or A/B testing, not as a guaranteed replacement for human judgment. Use top-K model outputs as a high-signal shortlist, and validate selected candidates with experiments or editorial review.
- What we do NOT claim: We do not claim causal uplift from applying the model, nor that the model's suggestions will increase traffic or revenue without follow-up testing. Observed correlations may be driven by confounding factors (visibility, historic investment, topic mix).

## Self-check

Before you submit, confirm each line honestly:

- [-] Every section above is filled — markdown thinking AND the code that backs it
- [-] The notebook runs top to bottom with no errors (Runtime → Run all)
- [-] No client names, URLs, or private queries anywhere
- [-] My claims use careful words: observed, measured, directional, decision-support
- [-] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.